# 20 · Opto in expert animals: PPC, ALM and the light itself

**What this shows.** Does silencing PPC on ~30 % of trials change expert behaviour on the Uniform
distribution? Per animal first, then WT vs HET across the cohort; then the two controls that make the
result interpretable — the same contrast on sham ("masking": blue light, no laser) sessions, which isolates
what the light alone does, and the same manipulation at ALM, which tests site specificity.

**What produced it.** An `opto_contrasts` run (`sc-reports opto-contrasts --all`, or `sc-reports battery`).
Each animal's `contrasts.csv` has one row per (kind, unit, stat):

| kind | what is compared | valid test |
|---|---|---|
| `within` | opto sessions: laser-on − laser-off trials | permutation (randomised per trial) and bootstrap |
| `within_masking` | sham sessions: flagged − unflagged trials | the light-only null |
| `between` | opto sessions − sham sessions, all trials | bootstrap only (sessions were not randomised) |
| `compensation` | laser-off trials of opto sessions − sham sessions | does behaviour change on the *other* trials? |
| `dod` | `within` − `within_masking` | laser effect beyond the light effect |
| `vs_ppc` | ALM sessions − PPC opto sessions | site specificity (ALM designs only) |

`unit` is `trials` (per-trial resampling) or `sessions` (whole-session resampling, always valid).
`group_rows.csv` / `group_tests.csv` hold the per-animal values and the WT-vs-HET rank tests.

```bash
sc-reports opto-contrasts --distribution Uniform --trial-class opto                 # PPC
sc-reports opto-contrasts --distribution Uniform --site alm_uni    # ALM control
```

In [ ]:
from nb_setup import *
banner()
run = open_run('opto_contrasts', OPTO_COHORT)
C = load_contrasts(run)
rows, tests = load_group(run)
genotype = C.drop_duplicates('animal').set_index('animal')['genotype']
STATS = ['mu', 'sigma', 'accuracy', 'side_bias', 'recency']
print('conditions:', sorted(set(zip(C.distribution, C.site, C.trial_class))))

## 1 · One animal: the contrasts from the library pipeline

`data/cohort.collect_sessions_ppc` picks the opto and sham sessions at one distribution;
`behaviour/contrasts.ppc_contrasts` computes every row of the table above with
`behav_utils.compute_delta_stat`. Few draws here, so it runs in seconds; the reports use 1000.

In [ ]:
from sound_categorisation.behaviour.contrasts import ppc_contrasts
from sound_categorisation.data.cohort import collect_sessions_ppc

exp = load_experiment()
aid = first_animal(C, 'het') if (genotype == 'het').any() else first_animal(C)
sets = collect_sessions_ppc(exp.animals[aid], 'Uniform')
print({k: len(v) for k, v in sets.items()}, 'sessions')
oc = ppc_contrasts(sets['opto'], sets['masking'], 'opto', names=STATS, n_boot=50, n_perm=50)
oc.table().query("kind in ('within', 'within_masking', 'dod')").round(3)

## 2 · Every animal: the laser-on − laser-off difference

`within`, trials unit: Δ with its bootstrap interval per animal and stat; WT in blue, HET in orange. A
WT animal has no opsin, so its `within` row is the light-delivery artefact at PPC — and it is not zero
for the criterion (`mu`): see §3.

In [ ]:
def forest(df, stat, ax, kind='within', unit='trials', title=None):
    d = df[(df.kind == kind) & (df.unit == unit) & (df.stat == stat)].copy()
    d['g'] = d.animal.map(genotype)
    d = d.sort_values(['g', 'animal'], ascending=[False, True]).reset_index(drop=True)
    for i, r in d.iterrows():
        c = GENOTYPE_COLOUR.get(r['g'], '0.5')
        ax.errorbar(r['diff'], i, xerr=[[r['diff'] - r['ci_lo']], [r['ci_hi'] - r['diff']]], fmt='o', color=c, ms=4)
    ax.axvline(0, color='0.6', lw=0.8)
    ax.set(yticks=range(len(d)), yticklabels=d.animal, title=title or f'{kind}: {stat}')
    return d

uni = C[(C.distribution == 'Uniform') & (C.site == 'ppc') & (C.trial_class == 'opto')]
fig, axes = plt.subplots(1, len(STATS), figsize=(3.2 * len(STATS), 0.35 * uni.animal.nunique() + 1.5), sharey=True)
for ax, st in zip(axes, STATS):
    forest(uni, st, ax)
plt.tight_layout()

## 3 · The light itself: the light-artefact report

`sc-reports light-artefact` contrasts light-on against light-off trials on every light-carrying session
set — `ppc_opto`, `ppc_sham` (blue light, no laser), `alm_uni`, `alm_bi` — with the same engine as the
opto contrasts, plus a signal-detection view: `criterion` (positive = biased toward A) and `dprime`. For a
WT animal every set is light-only. A criterion shift with intact d′ on `ppc_sham` that is absent on the
ALM sets is a site-specific light effect; `site_dod` (`ppc_sham − alm`) is its per-animal test.

```bash
sc-reports light-artefact --cohort opto1-cohort [--distribution Uniform]
```


In [ ]:
lrun = open_run('light_artefact', OPTO_COHORT)
LA = load_light_artefact(lrun, 'Uniform')
LC = LA['contrasts']
sets = [s for s in ('ppc_opto', 'ppc_sham', 'alm_uni', 'alm_bi') if s in set(LC.kind)]
fig, axes = plt.subplots(1, len(sets) + 1, figsize=(3.3 * (len(sets) + 1), 0.35 * LC.animal.nunique() + 1.6), sharey=True, squeeze=False)
for ax, name in zip(axes[0], sets):
    forest(LC, 'criterion', ax, kind=name, title=f'{name}: Δcriterion (on − off)')
if 'site_dod' in set(LC.kind):
    forest(LC, 'criterion', axes[0, -1], kind='site_dod', title='ppc_sham − alm: Δcriterion')
else:
    axes[0, -1].set_title('no ALM sets for site_dod'); axes[0, -1].axis('off')
plt.tight_layout()


Sensitivity should not move if this is a decision-rule effect: `dprime` on the same sets. Then the shift
per stimulus bin (a criterion shift moves P(B) by a similar amount at every stimulus), and the WT-vs-HET
fold per set.


In [ ]:
fig, axes = plt.subplots(1, len(sets), figsize=(3.3 * len(sets), 0.35 * LC.animal.nunique() + 1.6), sharey=True, squeeze=False)
for ax, name in zip(axes[0], sets):
    forest(LC, 'dprime', ax, kind=name, title=f"{name}: Δd′ (on − off)")
plt.tight_layout()

B = LA['bins']
fig, axes = plt.subplots(1, len(sets), figsize=(3.3 * len(sets), 3), sharey=True, squeeze=False)
for ax, name in zip(axes[0], sets):
    for aid, b in B[B['set'] == name].groupby('animal'):
        ax.plot(b['centre'], b['diff'], color=GENOTYPE_COLOUR.get(genotype.get(aid, 'unknown'), '0.5'), alpha=0.6, lw=1)
    ax.axhline(0, color='0.7', lw=0.6)
    ax.set(title=f'{name}: P(B) on − off per bin', xlabel='stimulus bin centre', ylim=(-0.4, 0.4))
plt.tight_layout()

gt = LA['group_tests']
(gt[gt.stat.isin(['criterion', 'dprime', 'mu'])][['kind', 'stat', 'n_a', 'n_b', 'median_a', 'median_b', 'p', 'min_p']]
   .sort_values(['kind', 'stat']).round(3)) if len(gt) else print('no group tests (one genotype present)')


## 4 · Across animals: WT vs HET

The valid unit for a genotype claim is the animal. `group_rows.csv` has one `within` value per animal and
stat; `group_tests.csv` the Mann–Whitney test between genotypes with `min_p`, the smallest p these
group sizes can produce — a p above `min_p` is uninformative, not null.

In [ ]:
g_rows = rows[(rows.distribution == 'Uniform') & (rows.site == 'ppc') & (rows.trial_class == 'opto') & (rows.kind == 'within')]
g_tests = tests[(tests.distribution == 'Uniform') & (tests.site == 'ppc') & (tests.trial_class == 'opto') & (tests.kind == 'within')]
fig, axes = plt.subplots(1, len(STATS), figsize=(2.6 * len(STATS), 3.2))
rng = np.random.default_rng(0)
for ax, st in zip(axes, STATS):
    d = g_rows[g_rows.stat == st]
    for j, g in enumerate(('wt', 'het')):
        v = d[d.group == g]['value'].to_numpy()
        ax.scatter(j + rng.uniform(-0.12, 0.12, len(v)), v, color=GENOTYPE_COLOUR[g], s=28, zorder=3)
        if len(v):
            ax.hlines(np.median(v), j - 0.25, j + 0.25, color=GENOTYPE_COLOUR[g], lw=2)
    ax.axhline(0, color='0.7', lw=0.6)
    t = g_tests[g_tests.stat == st]
    ptxt = f"p={t['p'].iloc[0]:.3f} (min {t['min_p'].iloc[0]:.3f})" if len(t) else 'no test'
    ax.set(xticks=[0, 1], xticklabels=['WT', 'HET'], title=f'{st}\n{ptxt}', xlim=(-0.6, 1.6))
plt.tight_layout()
g_tests[['stat', 'n_a', 'n_b', 'median_a', 'median_b', 'p', 'min_p']].round(3)

## 5 · Site specificity: the same manipulation at ALM

Unilateral and bilateral ALM sessions use the same trial structure; `vs_ppc` compares the ALM sessions
against the PPC opto sessions of the same animal. If the WT criterion shift is specific to light at PPC,
it should be absent at ALM.

In [ ]:
alm = C[C.site.str.startswith('alm') & (C.distribution == 'Uniform') & (C.trial_class == 'opto')]
if len(alm):
    sites = sorted(alm.site.unique())
    fig, axes = plt.subplots(1, 2 * len(sites), figsize=(5.5 * len(sites), 0.35 * alm.animal.nunique() + 1.5), sharey=True, squeeze=False)
    k = 0
    for site in sites:
        a = alm[alm.site == site]
        forest(a, 'mu', axes[0, k], kind='within', title=f'{site}: laser-on − off (mu)'); k += 1
        forest(a, 'mu', axes[0, k], kind='vs_ppc', unit='sessions', title=f'{site} − PPC sessions (mu)'); k += 1
    plt.tight_layout()
else:
    print('no ALM conditions in this run (sc-reports opto-contrasts --site alm_uni|bi)')

## 6 · The other trials, and after the laser

`compensation`: laser-off trials of opto sessions against sham sessions — do animals change strategy on
the unperturbed trials when a third of trials are perturbed? And `trial_class = post_opto`: the same `within`
contrast on the trial *after* a laser trial (sequential effects of inactivation).

In [ ]:
post = C[(C.distribution == 'Uniform') & (C.site == 'ppc') & (C.trial_class == 'post_opto')]
fig, axes = plt.subplots(1, 2, figsize=(8, 0.35 * uni.animal.nunique() + 1.5), sharey=True)
forest(uni, 'mu', axes[0], kind='compensation', unit='sessions', title='compensation: laser-off of opto − sham (mu)')
if len(post):
    forest(post, 'mu', axes[1], kind='within', title='post-opto trial − laser-off (mu)')
else:
    axes[1].set_title('no post_opto condition in this run'); axes[1].axis('off')
plt.tight_layout()